# 02 Clean the data

This notebook turns the raw HSE file into one clean file, `data/processed/d4_clean.csv`.
Every later notebook reads that file, so this one only needs to run once.

| Step | What it does |
|---|---|
| 1 | Load only the columns we need |
| 2 | Remove duplicate applications |
| 3 | Fix words that are joined by hyphens |
| 4 | Read the JSON columns (work history, education, skills) |
| 5 | Build one text for each applicant and one for each job |
| 6 | Fill missing numbers |
| 7 | Add English job titles |
| 8 | Mark the applicants we can evaluate |
| 9 | Save |

Takes about 2 to 4 minutes.

> **Stage 1, original study (restored).** This notebook is the step-by-step cleaning procedure, with its saved outputs.
> The same logic is available as a callable in `recommender/cleaning.py`, which `python -m recommender.pipeline run` uses to build the deployable model.
> Needs the raw D4 file at `data/raw/HSE_dataset(D4).csv` (see `data/README.md`).
> Re-running it rewrites `data/processed/d4_clean.csv`. If the bytes differ from the recorded run, `--reuse-prepared` will refuse them and you should do a full pipeline rebuild instead.

## 0. Setup

In [1]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Work from the project folder, even when Jupyter opens inside notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

RAW_FILE    = "data/raw/HSE_dataset(D4).csv"
LOOKUP_FILE = "data/lookup/job_titles_ru_en.csv"
OUT_FILE    = "data/processed/d4_clean.csv"

# If the raw file has another name, take the csv in data/raw/ that has "hse" in its name
if not Path(RAW_FILE).exists():
    found = [p for p in Path("data/raw").glob("*.csv") if "hse" in p.name.lower()]
    if not found:
        raise FileNotFoundError("Put the HSE csv file in the data/raw/ folder")
    RAW_FILE = str(found[0])

os.makedirs("data/processed", exist_ok=True)
os.makedirs("results", exist_ok=True)

steps = []   # a short log of each step, saved at the end for the report

print("project folder:", os.getcwd())
print("raw file      :", RAW_FILE)

project folder: /Users/anantaupadhyaya/Downloads/job-recommendation-salary-analyzer
raw file      : datasets/HSE_dataset(D4).csv


## 1. Load only the columns we need

The raw file is separated by `|`, not by commas. The Russian text contains many commas,
so reading it as a normal csv puts everything into one column.

**gender, age and birthday are left out on purpose**, so no model can use them.

In [2]:
SEP = "|"

USE_COLS = [
    "idCv", "idVacancy", "cv_status",
    # applicant side
    "positionName", "workExperienceList", "educationList",
    "skills_cv", "hardSkills_cv", "softSkills_cv",
    "experience", "salaryMin_cv", "localityName",
    # job side
    "vacancyName", "responsibilities", "positionRequirements",
    "skills_vacancy", "hardSkills_vacancy", "softSkills_vacancy",
    "professionalSphereName", "experienceRequirements", "salaryMin_vacancy",
    "regionName", "datePublished",
]

header = pd.read_csv(RAW_FILE, sep=SEP, nrows=0).columns
missing = [c for c in USE_COLS if c not in header]
USE_COLS = [c for c in USE_COLS if c in header]

print("columns in raw file :", len(header))
print("columns we load     :", len(USE_COLS))
print("not found           :", missing if missing else "none")

df = pd.read_csv(RAW_FILE, sep=SEP, usecols=USE_COLS, low_memory=False)

print("rows loaded         :", f"{len(df):,}")
steps.append(("1. Load", f"{len(df):,} rows, {len(USE_COLS)} of {len(header)} columns"))

columns in raw file : 91
columns we load     : 23
not found           : none
rows loaded         : 150,493


## 2. Remove duplicate applications

The same applicant and job pair can appear on several rows. We keep one row per pair.
When the copies disagree, we keep the invitation, because an invitation is something
an employer actually did.

`Приглашение` means invitation. `Отказ` means rejection.

In [3]:
df["is_positive"] = (df["cv_status"] == "Приглашение").astype(int)

before = len(df)

# invitations first, then keep the first row of each (applicant, job) pair
df = df.sort_values("is_positive", ascending=False)
df = df.drop_duplicates(subset=["idCv", "idVacancy"], keep="first")
df = df.reset_index(drop=True)

removed = before - len(df)
print(f"rows: {before:,} -> {len(df):,}")
print(f"removed {removed:,} duplicate rows ({removed / before:.1%})")
print(f"invitation rate now: {df['is_positive'].mean():.1%}")

steps.append(("2. Remove duplicates", f"{before:,} -> {len(df):,} rows ({removed / before:.1%} removed)"))

rows: 150,493 -> 93,755
removed 56,738 duplicate rows (37.7%)
invitation rate now: 15.9%


## 3. Fix words joined by hyphens

On the applicant side, spaces were replaced by hyphens when the data was exported:

```
positionName : Специалист-по-кадрам    (looks like 1 word)
vacancyName  : Специалист по кадрам    (3 words)
```

A text model sees no shared word between these two, even though they are the same job.
Putting the spaces back fixes this.

In [4]:
def fix_hyphens(text):
    # 'Специалист-по-кадрам' -> 'Специалист по кадрам'
    if not isinstance(text, str):
        return text
    return " ".join(text.replace("-", " ").split())

def shares_a_word(a, b):
    words_a = set(str(a).lower().split())
    words_b = set(str(b).lower().split())
    return len(words_a & words_b) > 0

sample = df.head(5000)
share_before = sample.apply(lambda r: shares_a_word(r["positionName"], r["vacancyName"]), axis=1).mean()

df["positionName"] = df["positionName"].map(fix_hyphens)
df["localityName"] = df["localityName"].map(fix_hyphens)

sample = df.head(5000)
share_after = sample.apply(lambda r: shares_a_word(r["positionName"], r["vacancyName"]), axis=1).mean()

print("example:", df["positionName"].head(3).tolist())
print(f"applicant title and job title share a word: {share_before:.1%} -> {share_after:.1%}")

steps.append(("3. Fix hyphens", f"title word overlap {share_before:.1%} -> {share_after:.1%}"))

example: ['Учитель английского языка', 'водитель автомобиля', 'Маляр']
applicant title and job title share a word: 20.0% -> 42.5%


## 4. Read the JSON columns

Work history, education and skills are stored as JSON text, for example:

```
workExperienceList : [{"jobTitle": "Бухгалтер", "demands": "Ведение учёта ..."}]
hardSkills_cv      : ["Работа в 1С", "Excel"]
```

We pull out the plain text so the models can read it.

In [5]:
def read_records(cell, keys):
    # [{"jobTitle": "A", "demands": "B"}, ...] -> "A B ..."
    try:
        items = json.loads(cell)
    except (TypeError, ValueError):
        return ""
    words = []
    for item in items:
        if isinstance(item, dict):
            for key in keys:
                value = item.get(key)
                if value and value != "None":
                    words.append(str(value))
    return " ".join(words)

def read_skills(cell):
    # ["Работа в 1С", "Excel"] -> "Работа в 1С Excel"
    try:
        items = json.loads(cell)
    except (TypeError, ValueError):
        return ""
    if not isinstance(items, list):
        return ""
    return " ".join(str(x) for x in items if x)

df["exp_text"] = df["workExperienceList"].map(lambda c: read_records(c, ["jobTitle", "demands"]))
df["edu_text"] = df["educationList"].map(lambda c: read_records(c, ["speciality", "qualification", "instituteName"]))

# join the three skill columns on each side into one text
df["cv_skills"] = ""
for col in ["skills_cv", "hardSkills_cv", "softSkills_cv"]:
    if col in df.columns:
        df["cv_skills"] = df["cv_skills"] + " " + df[col].map(read_skills)

df["job_skills"] = ""
for col in ["skills_vacancy", "hardSkills_vacancy", "softSkills_vacancy"]:
    if col in df.columns:
        df["job_skills"] = df["job_skills"] + " " + df[col].map(read_skills)

df["cv_skills"] = df["cv_skills"].str.strip()
df["job_skills"] = df["job_skills"].str.strip()

print(f"rows with work history text : {(df['exp_text'] != '').mean():.1%}")
print(f"rows with education text    : {(df['edu_text'] != '').mean():.1%}")
print()
print("example work history:", df.loc[df["exp_text"] != "", "exp_text"].iloc[0][:200])

rows with work history text : 89.6%
rows with education text    : 91.3%

example work history: мастер В соответствии с инструкцией


### How many applicants and jobs have skills?

We count unique applicants and unique jobs, because that is the unit the models work on.

In [6]:
applicant_has_skill = df.groupby("idCv")["cv_skills"].first() != ""
job_has_skill       = df.groupby("idVacancy")["job_skills"].first() != ""

print(f"applicants with at least one skill: {applicant_has_skill.mean():.1%}  ({applicant_has_skill.sum():,} people)")
print(f"jobs with at least one skill      : {job_has_skill.mean():.1%}  ({job_has_skill.sum():,} jobs)")

steps.append(("4. Skills", f"{applicant_has_skill.mean():.1%} of applicants, {job_has_skill.mean():.1%} of jobs have skills"))

applicants with at least one skill: 37.7%  (24,750 people)
jobs with at least one skill      : 16.6%  (7,285 jobs)


## 5. Build one text for each applicant and one for each job

- **cv_text** = desired position + work history + education
- **job_text** = job title + responsibilities + requirements

The title goes first because it is the strongest signal. Then we lowercase the text and
remove punctuation, keeping Russian letters, English letters and digits.

In [7]:
def join_parts(*parts):
    good = [str(p) for p in parts if isinstance(p, str) and p.strip() and p != "nan"]
    return " ".join(good)

def normalise(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zа-яё0-9\s]", " ", text)   # keep letters and digits only
    return " ".join(text.split())

df["cv_text"]  = [join_parts(a, b, c) for a, b, c in zip(df["positionName"], df["exp_text"], df["edu_text"])]
df["job_text"] = [join_parts(a, b, c) for a, b, c in zip(df["vacancyName"], df["responsibilities"], df["positionRequirements"])]

for col in ["cv_text", "job_text", "cv_skills", "job_skills"]:
    df[col] = df[col].map(normalise)

print("cv_text :", df["cv_text"].iloc[0][:150])
print("job_text:", df["job_text"].iloc[0][:150])
print()
print(f"average words  cv_text: {df['cv_text'].str.split().str.len().mean():.0f}   "
      f"job_text: {df['job_text'].str.split().str.len().mean():.0f}")

steps.append(("5. Build texts", "cv_text and job_text, lowercased, punctuation removed"))

cv_text : учитель английского языка преподаватель английского языка филолог дальневосточный федеральный университет двфу учитель английского языка обучение по ф
job_text: преподаватель кафедры иностранного языка ведение практических занятий по дисциплине русский язык как иностранный иностранный язык латинский язык стаж 

average words  cv_text: 73   job_text: 68


### One version per applicant and per job

An applicant appears on several rows (one per application). We make sure every row of the
same applicant carries the same text, and the same for jobs.

In [8]:
APPLICANT_COLS = ["positionName", "cv_text", "cv_skills", "experience", "salaryMin_cv", "localityName"]
JOB_COLS = ["vacancyName", "job_text", "job_skills", "professionalSphereName",
            "experienceRequirements", "salaryMin_vacancy", "regionName", "datePublished"]

for col in APPLICANT_COLS:
    df[col] = df.groupby("idCv")[col].transform("first")

for col in JOB_COLS:
    df[col] = df.groupby("idVacancy")[col].transform("first")

print("done")

done


## 6. Fill missing numbers

In [9]:
for col in ["experience", "experienceRequirements", "salaryMin_cv", "salaryMin_vacancy"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    n_missing = df[col].isna().sum()
    df[col] = df[col].fillna(df[col].median())
    print(f"{col:24s} filled {n_missing:,} missing values")

for col in ["professionalSphereName", "regionName", "localityName"]:
    df[col] = df[col].fillna("Unknown")

df["datePublished"] = pd.to_datetime(df["datePublished"], errors="coerce", format="mixed", utc=True)

steps.append(("6. Missing values", "numbers filled with the median, text filled with Unknown"))

experience               filled 0 missing values
experienceRequirements   filled 291 missing values
salaryMin_cv             filled 1 missing values
salaryMin_vacancy        filled 0 missing values


## 7. Add English job titles

The models use the Russian text. The English titles are only for us to read examples.
Titles that are not in the list stay in Russian.

In [10]:
def title_key(text):
    # match titles even if case, hyphens or spaces differ
    return " ".join(str(text).lower().replace("-", " ").split())

lookup = pd.read_csv(LOOKUP_FILE, encoding="utf-8-sig")
ru_to_en = {title_key(ru): en for ru, en in zip(lookup["ru"], lookup["en"])}

df["positionName_en"] = [ru_to_en.get(title_key(t), t) for t in df["positionName"]]
df["vacancyName_en"]  = [ru_to_en.get(title_key(t), t) for t in df["vacancyName"]]

cover_cv  = (df["positionName_en"] != df["positionName"]).mean()
cover_job = (df["vacancyName_en"] != df["vacancyName"]).mean()
print(f"titles in the list: {len(lookup):,}")
print(f"rows with English applicant title: {cover_cv:.1%}")
print(f"rows with English job title      : {cover_job:.1%}")

steps.append(("7. English titles", f"{cover_job:.1%} of job titles translated"))

titles in the list: 550
rows with English applicant title: 61.5%
rows with English job title      : 64.1%


## 8. Mark the applicants we can evaluate

We can only check a recommendation for someone who got at least one invitation,
because the invitation is the right answer we compare against.

In [11]:
df["n_invitations"] = df.groupby("idCv")["is_positive"].transform("sum")
df["in_eval_set"] = (df["n_invitations"] > 0).astype(int)

n_eval = df.loc[df["in_eval_set"] == 1, "idCv"].nunique()
print(f"applicants we can evaluate: {n_eval:,}")

steps.append(("8. Evaluation set", f"{n_eval:,} applicants with at least one invitation"))

applicants we can evaluate: 14,033


### Do people apply in their own region?

In [12]:
def region_key(text):
    # "Томская область, Колпашевский район, г Колпашево" -> "томская область"
    text = str(text).split(",")[0].lower().replace("-", " ")
    text = re.sub(r"^(город|г)\s+", "", text)
    return " ".join(text.split())

# localityName holds region, district and city; regionName holds only the region
same_region = [region_key(a) == region_key(b) for a, b in zip(df["localityName"], df["regionName"])]
same_region = np.mean(same_region)

print(f"applications in the applicant's own region: {same_region:.1%}")
steps.append(("Region check", f"{same_region:.1%} of applications are in the applicant's own region"))

applications in the applicant's own region: 82.6%


## 9. Save

In [13]:
KEEP = [
    "idCv", "idVacancy", "cv_status", "is_positive",
    # applicant
    "positionName", "positionName_en", "cv_text", "cv_skills",
    "experience", "salaryMin_cv", "localityName", "n_invitations", "in_eval_set",
    # job
    "vacancyName", "vacancyName_en", "job_text", "job_skills",
    "professionalSphereName", "experienceRequirements", "salaryMin_vacancy",
    "regionName", "datePublished",
]

clean = df[KEEP].sort_values(["idCv", "idVacancy"]).reset_index(drop=True)

# utf-8-sig lets Excel show the Russian letters correctly
clean.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")

# a small copy that opens quickly, for checking by eye
clean.sample(min(2000, len(clean)), random_state=42).to_csv(
    "data/processed/d4_clean_sample.csv", index=False, encoding="utf-8-sig")

print(f"saved {OUT_FILE}")
print(f"  {len(clean):,} rows x {clean.shape[1]} columns, {os.path.getsize(OUT_FILE) / 1e6:.0f} MB")
print(f"  applicants {clean['idCv'].nunique():,}   jobs {clean['idVacancy'].nunique():,}")
print(f"  invitation rate {clean['is_positive'].mean():.1%}")

saved clean/d4_clean.csv
  93,755 rows x 22 columns, 274 MB
  applicants 65,714   jobs 43,756
  invitation rate 15.9%


## Cleaning report

In [14]:
report = pd.DataFrame(steps, columns=["Step", "Result"])
report.to_csv("results/cleaning_report.csv", index=False, encoding="utf-8-sig")
print("saved results/cleaning_report.csv")
report

saved results/cleaning_report.csv


,Step,Result
0,1. Load,"150,493 rows, 23 of 91 columns"
1,2. Remove duplicates,"150,493 -> 93,755 rows (37.7% removed)"
2,3. Fix hyphens,title word overlap 20.0% -> 42.5%
3,4. Skills,"37.7% of applicants, 16.6% of jobs have skills"
4,5. Build texts,"cv_text and job_text, lowercased, punctuation ..."
5,6. Missing values,"numbers filled with the median, text filled wi..."
6,7. English titles,64.1% of job titles translated
7,8. Evaluation set,"14,033 applicants with at least one invitation"
8,Region check,82.6% of applications are in the applicant's o...
